# Case Study: Property, Plant, and Equipment (PP&E) Data Analysis

## Introduction: Translating Business Logic into Spreadsheet Architecture

In the preliminary sections of Unit 4, we noted that spreadsheet analytics is a process of reasoning rather than haphazard clicking. You learned how Microsoft Excel operates under the hood, how plain-text CSV files transmit bulk enterprise data, and how SQL's seven fundamental operations map directly onto spreadsheet functions.

Now, we put this framework into action through our first major case study: auditing a company's Property, Plant, and Equipment (PP&E) account.

As an accountant, financial analyst, or auditor, you will rarely receive perfectly formatted balance sheet schedules. Instead, client Enterprise Resource Planning (ERP) systems extract raw transaction logs containing thousands of unformatted ledger debit and credit entries. 

In this case study, we analyze a dataset of 1,430 general ledger transaction amounts hitting a firm's PP&E account. Through this case, you will learn how to:
1. Distinguish additions to the account from disposals or depreciation.
2. Answer executive financial questions using both classic conditional functions (`COUNTIF`, `SUMIF`, `AVERAGEIF`) and modern **dynamic array functions** (`FILTER`, `SORT`, `TAKE`).
3. Build clean, auditable models using **named ranges**.
4. Enrich raw transactional logs by generating random transaction IDs and simulated dates using date serial integers and `RANDBETWEEN`.

---

## Understanding the Accounting Data: Debits vs. Credits in General Ledger Logs

### Data Preview

Download the data file by clicking <u>[this link](data/PPEdata.csv).</u> The data file is a CSV file that contains only one column of numbers. Each number is from a journal entry that hit the Property, Plant, and Equipment account. Following is a preview of the data, showing the first ten rows.

In [2]:
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import HTML, display

In [3]:
df = pd.read_csv('data/PPEdata.csv')
display(HTML(df.head().to_html(index=False)))

Amount
-22539
58277
-37200
175266
-21190


### Understanding the Data

Before opening Excel or writing a single formula, you must understand the accounting rules governing the data-generating process.

In general ledger accounting, Property, Plant, and Equipment is an *asset account* on the balance sheet. Under double-entry bookkeeping:
* **Positive Amounts (Debits):** Represent asset purchases, capital additions, or improvements that increase the PP&E account balance (e.g., purchasing a new manufacturing machine for `$58,277` or a facility expansion for `$175,266`).
* **Negative Amounts (Credits):** Represent transactions that reduce the PP&E account balance—such as asset disposals, retirements, or periodic depreciation postings (e.g., crediting PP&E by `-$22,539` or `-$37,200`).

```{warning}
**The Auditor's Perspective: Interpreting Signed Ledger Entries**  
Inexperienced accountants often get confused by negative numbers in transaction logs. In a PP&E ledger export, a negative sign does **not** indicate an error or bad debt. It simply reflects a credit entry. Recognizing that `Amount > 0` defines capital purchases is the key filter condition required for all subsequent financial calculations.
```

---

## Answering Core Financial Questions (SQL vs. Excel)

Management and the audit team request answers to five specific financial questions regarding the PP&E account. Below, we examine the logical process for answering each question, contrasting SQL queries with both crude and modern Excel approaches.

### Question 1: How many total transactions impacted the PP&E account?

* **Goal:** Determine the total volume of journal entries posted to the account during the period.
* **Strategy:** Since each row represents an amount from a transaction, we must count the number of rows in the data.
* **SQL Query:**
  ```sql
  SELECT COUNT(*) AS TotalTransactions
  FROM PPE
  ```
* **Excel Solution:** Use the static `COUNT` function over the entire range (`A2:A1431`):
  ```sql
  =COUNT(A2:A1431)
  ```
* **Result:** 1,430 transactions.

### Question 2: How many individual asset purchases (additions) were made?

* **Goal:** Determine how many assets were purchased.
* **Strategy:** Debits are almost certainly the result of asset purchases. As we noted earlier, positive numbers are debits. Therefore, count only positive transactions (`Amount > 0`).
* **The "Crude" Excel Way:** Turn on the GUI AutoFilter button, filter the column for values greater than 0, copy and paste the filtered rows to a new sheet, and highlight the column to read the count in the status bar.
  *(Why this is crude: It breaks the audit trail, creates duplicated static files, and does not update when new data is added.)*
* **The Modern Excel Solution:** Use the conditional `COUNTIF` function:
  ```sql
  =COUNTIF(A2:A1431, ">0")
  ```
  *Note on `COUNTIF` Syntax:* The first argument is the evaluation range (`A2:A1431`), and the second argument is the text criterion inside quotes (`">0"`).
* **SQL Query:**
  ```sql
  SELECT COUNT(*) AS PurchaseCount
  FROM PPE
  WHERE Amount > 0
  ```
* **Result:** 718 asset purchases.

### Question 3: What was the total dollar amount of additions to PP&E?

* **Goal:** Calculate the total capital expenditure (CapEx) spent on acquiring new assets.
* **Strategy:** Filter the data to only positive numbers, then sum those positive numbers.
* **SQL Query:**
  ```sql
  SELECT SUM(Amount) AS TotalAdditions
  FROM PPE
  WHERE Amount > 0
  ```
* **Excel Solution:** Use the conditional `SUMIF` function:
  ```sql
  =SUMIF(A2:A1431, ">0")
  ```
  *(Optional 3-Argument Syntax:* When the range to evaluate and the range to sum are different, `SUMIF` takes three arguments: `=SUMIF(CategoryRange, "Criteria", SumRange)`.*)*
* **Result:** \$78,773,067.00.

### Question 4: What was the average purchase price of new assets?

* **Goal:** Determine the average purchase price of newly acquired assets.
* **Strategy:** Filter the data to only positive numbers, then take the average of those numbers.
* **SQL Query:**
  ```sql
  SELECT AVG(Amount) AS AverageAddition
  FROM PPE
  WHERE Amount > 0
  ```
* **Excel Solution:** Use the conditional `AVERAGEIF` function:
  ```sql
  =AVERAGEIF(A2:A1431, ">0")
  ```
* **Result:** \$109,711.79.

### Question 5: How much additional depreciation expense will these acquisitions generate in the current year?

* **Goal:** Determine the amount of *incremental* depreciation expense that the company will incur from the newly acquired assets.
* **Accounting Assumptions:**
  1. All new assets have an estimated useful life of 10 years.
  2. All acquisitions occurred evenly throughout the fiscal year (middle-of-the-year / half-year convention). Under the half-year convention, new assets generate only 6 months (50%) of full-year straight-line depreciation in their acquisition year.
* **Mathematical Model:**
  $\text{Current-Year Depreciation} = \frac{\text{Total Additions}}{\text{Useful Life (10 Years)}} \times 0.5 = \frac{\text{Total Additions}}{20}$
* **SQL Query:**
  ```sql
  SELECT SUM((Amount / 10.0) / 2.0) AS CurrentYearDepreciation
  FROM PPE
  WHERE Amount > 0
  ```
* **Excel Solution:** Divide the output of `SUMIF` by 20:
  ```sql
  =(SUMIF(A2:A1431, ">0") / 10) / 2
  ```
* **Result:** \$3,938,653.35.

### Summary Results

| Question | Accounting Metric | Filter Condition | Excel Formula | Calculated Result |
| :--- | :--- | :--- | :--- | :--- |
| Q1 | Total Transactions | None | `=COUNT(A2:A1431)` | 1,430 |
| Q2 | Asset Purchase Count | `Amount > 0` | `=COUNTIF(A2:A1431, ">0")` | 718 |
| Q3 | Total CapEx Additions | `Amount > 0` | `=SUMIF(A2:A1431, ">0")` | \$78,773,067.00 |
| Q4 | Average Addition Cost | `Amount > 0` | `=AVERAGEIF(A2:A1431, ">0")` | \$109,711.79 |
| Q5 | Y1 Depreciation Expense | `Amount > 0` | `=(SUMIF(A2:A1431, ">0")/10)/2` | \$3,938,653.35 |

---

## Modern Excel Features (Dynamic Arrays & Named Ranges)

While `COUNTIF`, `SUMIF`, and `AVERAGEIF` are huge improvements over manual GUI filtering, modern Excel power users leverage **dynamic array functions** and **named ranges** to build self-documenting formulas that never break.

### 1. Clean Formulas with Named Ranges

Instead of typing cryptic cell coordinates like `A2:A1431` inside every formula, assign a descriptive **named range** to the raw data block:
* Select `A2:A1431` and name the range **`PPE`** in the Name Box (top-left of the formula bar).
* Create a dynamic subset range named **`Additions`** using the `FILTER` function:
  ```
  Additions = FILTER(PPE, PPE > 0)
  ```

Now, your financial formulas become readable plain-English sentences:
```
=COUNT(Additions)      --> 718  
=SUM(Additions)        --> $78,773,067.00  
=AVERAGE(Additions)    --> $109,711.79  
=MIN(Additions)        --> $294.00  
=MAX(Additions)        --> $219,471.00  
```

### 2. The `FILTER` Dynamic Array Function

The `FILTER` function evaluates a condition across an entire array and **spills** the matching results into neighboring empty cells below:
```
=FILTER(range, include)
```
* `range`: The block of data to return (e.g., `A2:A1431`).
* `include`: A boolean array returning `TRUE` or `FALSE` for every row (e.g., `PPE > 0`).

### 3. Ranking Outliers: Nesting `SORT` and `TAKE`

Suppose the CFO asks: *"Show me the top 5 largest asset purchases in 2024 for executive capital review."*

Traditional accountants sort the worksheet manually and copy the top 5 rows. Excel power users nest the **`TAKE`**, **`SORT`**, and **`FILTER`** dynamic array functions in a single cell:

```
=TAKE(SORT(FILTER(PPE, PPE > 0), 1, -1), 5)
```
Or, using our named range:
```
=TAKE(SORT(Additions, 1, -1), 5)
```

Deconstructing the formulas:
1. **`FILTER(PPE, PPE > 0)`**: Isolates the 718 positive additions.
2. **`SORT(..., 1, -1)`**: Sorts the filtered array by column 1 in descending order (`-1` = descending; `1` = ascending).
3. **`TAKE(..., 5)`**: Extracts the first 10 rows from the sorted array!

The result:

In [9]:
df2Display = df[df['Amount'] > 0].sort_values(['Amount'], ascending=False).head(5)
df2Display['Amount'] = df2Display['Amount'].apply(lambda x: f'${x:,.0f}')
display(HTML(df2Display.to_html(index=False)))

Amount
"$219,471"
"$219,311"
"$218,634"
"$217,910"
"$217,531"


---

## Data Enrichment: Random IDs and Acquisition Dates

Our raw CSV file contains only a single column of monetary amounts. To perform full forensic audit analytics—such as checking monthly acquisition trends or matching entries across ledger systems—we need transaction IDs and transaction dates.

Normally, transaction data will contain these attributes. However, since our data does not contain those columns, we can *simulate* them by using Excel's random number functinos. This will help you understand how Excel handles numbers and dates.

### 1. Generating Random 10-Digit Transaction IDs (`RANDBETWEEN`)

To generate random 10-digit transaction identifiers:
```
=RANDBETWEEN(1000000000, 9999999999)
```
`RANDBETWEEN(bottom, top)` generates uniformly distributed random integers between the specified bounds.

```{warning}
**Critical Rule: Freezing Volatile Functions with Copy / Paste Special**  
`RANDBETWEEN` is a **volatile function**. Every time you edit any cell in Excel or force a recalculation, `RANDBETWEEN` will generate brand-new numbers!
To freeze your generated transaction IDs into permanent audit records:
1. Highlight the column of formulas.
2. Press `Ctrl + C` (Copy).
3. Right-click and choose **Paste Special -> Values** (`Ctrl + Alt + V -> V`). This replaces the volatile `=RANDBETWEEN(...)` formula with static integer values!
```

### 2. Generating Random Dates (Understanding Dates as Serial Numbers)

Excel stores dates as numbers. The number indicates how many days have elapsed since January 1, 1900:
* `1/1/1900` = Serial Integer `1`
* `1/1/2024` = Serial Integer `45292`
* `12/31/2024` = Serial Integer `45657`

Because dates are stored as integers, you can generate random acquisition dates across fiscal year 2024 using `RANDBETWEEN`:
```
=RANDBETWEEN(DATE(2024, 1, 1), DATE(2024, 12, 31))
```
Once generated, apply Currency and Date number formatting (`Ctrl + 1`) to render the underlying numbers visually as `$125,500.00` and `2024-06-15`.

### 3. Sample Result

Adding transaction ID's and dates will result in data that looks like the following. Your result will differ since the random numbers that you generate will be different.

| Transaction ID | Acquisition Date | Ledger Amount |
| :--- | :--- | :--- |
| 8492019482 | 2024-02-14 | \$175,266 |
| 1092837492 | 2024-04-01 | \$(22,539) |
| 9283741029 | 2024-06-20 | \$196,703 |
| 3847291023 | 2024-09-11 | \$28,397 |
| 5739201948 | 2024-11-05 | \$(37,200) |

---

## Summary

This case study demonstrates how a single column of raw transaction amounts can be transformed into an executive financial analysis using modern Excel functions.

### Key Case Study Takeaways

1. **Debit vs. Credit Logic:** Positive amounts (`> 0`) represent capital asset acquisitions; negative amounts represent credits, retirements, or disposals.
2. **Conditional Summaries:** Use `COUNTIF`, `SUMIF`, and `AVERAGEIF` for targeted category filtering without destroying raw data structure.
3. **Dynamic Array Functions:** `FILTER`, `SORT`, and `TAKE` allow you to dynamically extract and rank subsets of data without manual GUI filtering or copy-pasting.
4. **Named Ranges:** Naming raw data blocks (`PPE`, `Additions`) makes complex financial formulas readable, auditable, and self-documenting.
5. **Freezing Volatile Functions:** Always use **Copy -> Paste Special Values** after generating random transaction IDs or dates with `RANDBETWEEN`.

### Summary of Functions Used in the PP&E Case Study

| Function | Syntax Example | Primary Accounting / Analytics Role |
| :--- | :--- | :--- |
| **`COUNT`** | `=COUNT(Range)` | Counts total numeric transaction entries. |
| **`COUNTIF`** | `=COUNTIF(Range, ">0")` | Counts specific subset records (e.g., purchase count). |
| **`SUMIF`** | `=SUMIF(Range, ">0")` | Sums dollar amounts matching a condition (e.g., CapEx additions). |
| **`AVERAGEIF`** | `=AVERAGEIF(Range, ">0")` | Computes average cost per purchase. |
| **`FILTER`** | `=FILTER(PPE, PPE > 0)` | Spills dynamic filtered array of positive asset purchases. |
| **`SORT`** | `=SORT(Range, 1, -1)` | Sorts dynamic arrays in ascending or descending order. |
| **`TAKE`** | `=TAKE(SortedArray, 10)` | Extracts the Top N (or Bottom N) outlier rows. |
| **`RANDBETWEEN`** | `=RANDBETWEEN(bottom, top)` | Generates random integers for transaction IDs or date serials. |